In [1]:
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from scipy.ndimage import convolve, binary_dilation
from pathlib import Path
from tqdm.auto import tqdm
import random
import matplotlib.pyplot as plt

DATA_DIR = Path.cwd().parent.parent / "data" / "hyperspectral_oil_spill"
device = "cuda" if torch.cuda.is_available() else "cpu"

train_files = [f for f in DATA_DIR.glob("*.mat") if f.stem not in ["GM01", "GM02"]]
test_files = [f for f in DATA_DIR.glob("*.mat") if f.stem in ["GM01", "GM02"]]

def get_adaptive_clean_bands(train_files):
    mask = np.array([[1, -2,  1], 
                     [-2, 4, -2], 
                     [1, -2,  1]], dtype=float)
    all_scores = []
    
    for file_path in tqdm(train_files, desc="Evaluating Spectral Noise"):
        img = sio.loadmat(file_path)["img"]
        H, W, C = img.shape
        scores = []
        
        for b in range(C):
            total_noise = 0.0
            # Stripe processing to prevent RAM exhaustion
            for r in range(1, H - 1, 64):
                end = min(r + 64, H - 1)
                stripe = img[r-1:end+1, :, b].astype(float)
                total_noise += np.abs(convolve(stripe, mask)[1:-1, 1:-1]).sum()
            
            # Variance estimation formula from the HOSD isolation forest methodology
            sigma_n = total_noise * np.sqrt(np.pi / 2) / (6 * (H - 2) * (W - 2))
            scores.append(sigma_n)
            
        all_scores.append(scores)
        
    avg_sigma = np.mean(all_scores, axis=0)
    
    # Adaptive threshold: sigma_n < (1 / 2 * I_N) * sum(sigma_n)
    threshold = np.mean(avg_sigma) / 2.0
    clean_bands = np.flatnonzero(avg_sigma < threshold)
    return clean_bands

CLEAN_BANDS = get_adaptive_clean_bands(train_files)
print(f"Selected {len(CLEAN_BANDS)} clean bands using adaptive thresholding.")

Evaluating Spectral Noise:   0%|          | 0/16 [00:00<?, ?it/s]

Selected 42 clean bands using adaptive thresholding.


In [2]:
class HSIPatchDataset(Dataset):
    def __init__(self, file_list, clean_bands, patch_size=11, augment=False):
        self.patch_size = patch_size
        self.augment = augment
        self.arrays = []
        self.items = []
        pad = patch_size // 2
        
        for idx, file_path in enumerate(tqdm(file_list, desc="Mining Full Dataset")):
            mat = sio.loadmat(file_path)
            img = mat["img"][:, :, clean_bands].astype(np.float32)
            img = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8)
            img_padded = np.pad(img, ((pad, pad), (pad, pad), (0, 0)), mode='symmetric')
            self.arrays.append(img_padded)
            
            gt = mat["map"].astype(np.int64)
            oil_mask = (gt == 1)
            water_mask = (gt == 0)
            
            oil_coords = np.argwhere(oil_mask)
            n_oil = len(oil_coords)
            
            # Skip scenes with zero oil to prevent sampling errors
            if n_oil == 0:
                continue
            
            dilated_oil = binary_dilation(oil_mask, iterations=3)
            hard_water_mask = dilated_oil & water_mask
            hard_water_coords = np.argwhere(hard_water_mask)
            
            easy_water_mask = water_mask & ~hard_water_mask
            easy_water_coords = np.argwhere(easy_water_mask)
            
            np.random.shuffle(hard_water_coords)
            np.random.shuffle(easy_water_coords)
            
            # Match the exact number of oil pixels to maintain strict 50:50 balance
            water_sampled = np.vstack((hard_water_coords, easy_water_coords))[:n_oil]
            
            for r, c in oil_coords:
                self.items.append((idx, r, c, 1.0))
            for r, c in water_sampled:
                self.items.append((idx, r, c, 0.0))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        file_idx, r, c, label = self.items[idx]
        patch = self.arrays[file_idx][r:r+self.patch_size, c:c+self.patch_size, :]
        
        if self.augment:
            k = random.choice([0, 1, 2, 3])
            if k > 0:
                patch = np.rot90(patch, k=k, axes=(0, 1))
            if random.random() > 0.5:
                patch = np.fliplr(patch)
            if random.random() > 0.5:
                patch = np.flipud(patch)
                
        patch_tensor = torch.from_numpy(patch.copy().transpose(2, 0, 1))
        return patch_tensor, torch.tensor(label, dtype=torch.float32)

# num_workers=0 ensures stability in Jupyter environments
train_dataset = HSIPatchDataset(train_files, CLEAN_BANDS, augment=True)
val_dataset = HSIPatchDataset(test_files, CLEAN_BANDS, augment=False)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=0)

Mining Full Dataset:   0%|          | 0/16 [00:00<?, ?it/s]

Mining Full Dataset:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
class CoTNetLayer(nn.Module):
    def __init__(self, dim, kernel_size=3):
        super().__init__()
        self.kernel_size = kernel_size
        self.key_embed = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=kernel_size, padding=1, bias=False),
            nn.BatchNorm2d(dim),
            nn.ReLU(inplace=True)
        )
        self.value_embed = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(dim)
        )
        factor = 4
        self.attention_embed = nn.Sequential(
            nn.Conv2d(2 * dim, 2 * dim // factor, 1, bias=False),
            nn.BatchNorm2d(2 * dim // factor),
            nn.ReLU(inplace=True),
            nn.Conv2d(2 * dim // factor, kernel_size * kernel_size * dim, 1)
        )

    def forward(self, x):
        bs, c, h, w = x.shape
        k1 = self.key_embed(x) 
        v = self.value_embed(x).view(bs, c, -1) 
        y = torch.cat([k1, x], dim=1) 
        att = self.attention_embed(y) 
        att = att.reshape(bs, c, self.kernel_size * self.kernel_size, h, w)
        att = att.mean(2, keepdim=False).view(bs, c, -1) 
        k2 = F.softmax(att, dim=-1) * v 
        k2 = k2.view(bs, c, h, w)
        return k1 + k2 

class True_SSTNet(nn.Module):
    def __init__(self, in_bands, embed_dim=128):
        super().__init__()
        # Exact 7x1x1 3D Spectral Convolution from Paper 2
        self.spectral_conv = nn.Sequential(
            nn.Conv3d(1, embed_dim, kernel_size=(7, 1, 1), padding=(3, 0, 0), bias=False),
            nn.BatchNorm3d(embed_dim),
            nn.ReLU(inplace=True),
            # Compress remaining band dimension down to 1 for the 2D spatial layers
            nn.Conv3d(embed_dim, embed_dim, kernel_size=(in_bands, 1, 1), bias=False),
            nn.BatchNorm3d(embed_dim),
            nn.ReLU(inplace=True)
        )
        
        # Stacked Spatial Contextual Transformers
        self.cot1 = CoTNetLayer(dim=embed_dim)
        self.cot2 = CoTNetLayer(dim=embed_dim)
        self.pool = nn.AdaptiveAvgPool2d(1)
        
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x shape: (B, Bands, H, W) -> add channel dim for 3D Conv: (B, 1, Bands, H, W)
        x = x.unsqueeze(1) 
        x = self.spectral_conv(x) 
        # Squeeze out the reduced band dimension: (B, embed_dim, 1, H, W) -> (B, embed_dim, H, W)
        x = x.squeeze(2) 
        
        x = self.cot1(x)
        x = self.cot2(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x).squeeze(1)

model = True_SSTNet(in_bands=len(CLEAN_BANDS)).to(device)

In [4]:
from torchinfo import summary

summary(model, input_size=(1, len(CLEAN_BANDS), 11, 11), 
        col_names=["input_size", "output_size", "num_params", "mult_adds"])

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Mult-Adds
True_SSTNet                              [1, 42, 11, 11]           [1]                       --                        --
├─Sequential: 1-1                        [1, 1, 42, 11, 11]        [1, 128, 1, 11, 11]       --                        --
│    └─Conv3d: 2-1                       [1, 1, 42, 11, 11]        [1, 128, 42, 11, 11]      896                       4,553,472
│    └─BatchNorm3d: 2-2                  [1, 128, 42, 11, 11]      [1, 128, 42, 11, 11]      256                       256
│    └─ReLU: 2-3                         [1, 128, 42, 11, 11]      [1, 128, 42, 11, 11]      --                        --
│    └─Conv3d: 2-4                       [1, 128, 42, 11, 11]      [1, 128, 1, 11, 11]       688,128                   83,263,488
│    └─BatchNorm3d: 2-5                  [1, 128, 1, 11, 11]       [1, 128, 1, 11, 11]       256                       256


In [ ]:
criterion = nn.BCEWithLogitsLoss() 
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
EPOCHS = 15

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for X, y in pbar:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    avg_train_loss = total_loss / len(train_loader)
    
    model.eval()
    all_preds, all_targets = [], []
    
    with torch.inference_mode():
        for X, y in val_loader:
            probs = torch.sigmoid(model(X.to(device))).cpu().numpy()
            all_preds.extend(probs)
            all_targets.extend(y.numpy())
            
    preds_binary = (np.array(all_preds) > 0.5).astype(int)
    auc = roc_auc_score(all_targets, all_preds)
    f1 = f1_score(all_targets, preds_binary)
    precision = precision_score(all_targets, preds_binary, zero_division=0)
    recall = recall_score(all_targets, preds_binary, zero_division=0)
    
    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | AUC: {auc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

Epoch 1/15:   0%|          | 0/5870 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.4535 | AUC: 0.6752 | Precision: 0.8943 | Recall: 0.2740 | F1: 0.4195


Epoch 2/15:   0%|          | 0/5870 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.3944 | AUC: 0.6841 | Precision: 0.8236 | Recall: 0.3890 | F1: 0.5284


Epoch 3/15:   0%|          | 0/5870 [00:00<?, ?it/s]

...

Epoch 13 | Train Loss: 0.3153 | AUC: 0.7055 | Precision: 0.7654 | Recall: 0.4285 | F1: 0.5494


Epoch 14/15:   0%|          | 0/5870 [00:00<?, ?it/s]

Epoch 14 | Train Loss: 0.3105 | AUC: 0.6916 | Precision: 0.7503 | Recall: 0.4166 | F1: 0.5358


Epoch 15/15:   0%|          | 0/5870 [00:00<?, ?it/s]

Epoch 15 | Train Loss: 0.3067 | AUC: 0.6963 | Precision: 0.7234 | Recall: 0.4963 | F1: 0.5888


In [ ]:
# 147 Mins to train
# Epoch 1 | Train Loss: 0.4535 | AUC: 0.6752 | Precision: 0.8943 | Recall: 0.2740 | F1: 0.4195
# Epoch 5 | Train Loss: 0.3566 | AUC: 0.6948 | Precision: 0.7383 | Recall: 0.4604 | F1: 0.5672
# Epoch 10 | Train Loss: 0.3282 | AUC: 0.6910 | Precision: 0.7296 | Recall: 0.4854 | F1: 0.5830
# Epoch 15 | Train Loss: 0.3067 | AUC: 0.6963 | Precision: 0.7234 | Recall: 0.4963 | F1: 0.588

# This validation dataset is all oil + surrounding water + water so that the validation dataset is 50-50 oil and water

In [ ]:
def get_enhanced_rgb(img_array, rgb_bands=[29, 19, 9]):
    """
    Extracts and enhances RGB bands from the raw hyperspectral image.
    Red, Green, Blue bands (approx ~650nm, ~550nm, ~480nm in AVIRIS)
    """
    rgb_raw = img_array[:, :, rgb_bands].astype(np.float32)
    rgb_clean = np.where(rgb_raw < 0, 0, rgb_raw)
    
    rgb_enhanced = np.zeros_like(rgb_clean)
    for c in range(3):
        channel = rgb_clean[:, :, c]
        p_low, p_high = np.percentile(channel[channel > 0], (2, 98))
        rgb_enhanced[:, :, c] = np.clip((channel - p_low) / (p_high - p_low + 1e-8), 0, 1)
        
    return rgb_enhanced

def plot_full_scene(model, file_path, clean_bands, device, patch_size=11):
    mat = sio.loadmat(file_path)
    img = mat["img"]
    gt = mat["map"]
    
    rgb_img = get_enhanced_rgb(img)
    
    img_clean = img[:, :, clean_bands].astype(np.float32)
    img_clean = (img_clean - np.min(img_clean)) / (np.max(img_clean) - np.min(img_clean) + 1e-8)
    
    pad = patch_size // 2
    img_padded = np.pad(img_clean, ((pad, pad), (pad, pad), (0, 0)), mode='symmetric')
    
    H, W = gt.shape
    predictions = np.zeros((H, W))
    
    # Trackers for full-scene metrics
    all_probs = []
    all_targets = []
    
    # 1. Collect all valid coordinates
    valid_coords = [(r, c) for r in range(H) for c in range(W) if gt[r, c] >= 0]
    
    # 2. Process in strict hardware-friendly batches
    batch_size = 256 
    
    model.eval()
    with torch.inference_mode():
        # Tqdm will now track total batches rather than jumping row-by-row
        for i in tqdm(range(0, len(valid_coords), batch_size), desc=f"Inferring {file_path.stem}"):
            batch_coords = valid_coords[i:i+batch_size]
            batch = []
            
            for r, c in batch_coords:
                patch = img_padded[r:r+patch_size, c:c+patch_size, :]
                batch.append(patch.transpose(2, 0, 1))
            
            # Push capped batch to GPU
            batch_tensor = torch.tensor(np.array(batch), dtype=torch.float32).to(device)
            probs = torch.sigmoid(model(batch_tensor)).cpu().numpy()
            
            for (r, c), prob in zip(batch_coords, probs):
                predictions[r, c] = 1 if prob > 0.5 else 0
                all_probs.append(prob)
                all_targets.append(gt[r, c])

    # Calculate and print metrics
    all_probs = np.array(all_probs)
    all_targets = np.array(all_targets)
    preds_binary = (all_probs > 0.5).astype(int)
    
    auc = roc_auc_score(all_targets, all_probs)
    precision = precision_score(all_targets, preds_binary, zero_division=0)
    recall = recall_score(all_targets, preds_binary, zero_division=0)
    f1 = f1_score(all_targets, preds_binary, zero_division=0)
    
    print(f"\n--- Full Scene Metrics for {file_path.stem} ---")
    print(f"AUC: {auc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n")

    # Plotting
    fig, axs = plt.subplots(1, 3, figsize=(18, 8))
    axs[0].imshow(rgb_img)
    axs[0].set_title(f"False-Color RGB ({file_path.stem})")
    
    axs[1].imshow(gt == 1, cmap='magma')
    axs[1].set_title("Ground Truth Mask")
    
    axs[2].imshow(predictions, cmap='magma')
    axs[2].set_title("Model Prediction")
    
    for ax in axs: ax.axis("off")
    plt.tight_layout()
    plt.show()

plot_full_scene(model, test_files[0], CLEAN_BANDS, device)

In [ ]:
erw?